In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))
import pandas as pd


In [2]:
import src.preference_factors.build_datasets as bd


In [ ]:
from src.FamaMacBeth.betas import compute_rolling_betas
from src.FamaMacBeth.pipeline import run_full_factor_pipeline

merged_df, full_merged_df = bd.build_all_dataset(
    returns_file="sp500_returns_daily_with_tickers.csv",
    market_caps_file="sp500_market_caps_daily.csv",
    ff_factors_file="ff_factors_daily.csv",
    frequency="daily"
)

ebc = merged_df["EBC"]
cw_ebc = merged_df["CW-EBC"]


In [ ]:
ebc = ebc.to_frame(name="EBC")
cw_ebc = cw_ebc.to_frame(name="CW-EBC")

In [ ]:
asset_returns = bd.build_returns_dataset("sp500_returns_daily_with_tickers.csv", frequency="daily")
factor_returns = pd.concat([ebc, cw_ebc], axis=1).dropna()
factor_returns.columns = ["EBC", "CW-EBC"]

asset_returns, factor_returns = asset_returns.align(factor_returns, join="inner", axis=0)

betas_ebc = compute_rolling_betas(asset_returns, ebc, rolling_window=252, min_obs=10)
betas_cw_ebc = compute_rolling_betas(asset_returns, cw_ebc, rolling_window=252, min_obs=10)



In [ ]:
results = run_full_factor_pipeline(
    beta1_df=betas_ebc,
    beta2_df=betas_cw_ebc,
    sp500=asset_returns,
    EBC=factor_returns["EBC"],
    Cap_EBC=factor_returns["CW-EBC"],
    n_q1=3,
    n_q2=3,
    min_assets_per_cell=3,
    f1_name="EBC",
    f2_name="CW-EBC",
    frequency = 'daily'
)

results["mean_grid"]

In [ ]:
results["portrets_wide"][:5]

In [ ]:
from pathlib import Path
import src.FamaMacBeth.backtest as backtest
'''(
    load_monthly_inputs,
    backtest_quantile_portfolios,
    compute_excess_returns,
)
'''
import src.FamaMacBeth.plots as plots
'''(
    plot_cum_log_returns,
    plot_expected_return_grid,
)
'''
from src.FamaMacBeth.plots import plot_mean_grid

In [ ]:
# ============================
# USER PARAMETERS
# ============================
start_date = "1970-01-01"
use_excess_returns = True

# ============================
# REQUIRED INPUTS FROM PIPELINE
# ============================
members_df = results["members"]
n_q1, n_q2 = results["mean_grid"].shape

# ============================
# LOAD RETURNS / CAPS / RF
# ============================
DATA_RAW_DIR = Path.cwd().parent / "data" / "raw"
sp_500, sp_caps, rf = backtest.load_daily_inputs(DATA_RAW_DIR)

In [ ]:
# ============================
# RUN BACKTEST
# ============================
quantile_portfolios_returns = backtest.backtest_quantile_portfolios(
    members_df=members_df,
    returns_df=sp_500,
    caps_df=sp_caps,
    n_q1=n_q1,
    n_q2=n_q2,
    weight="cap",
)

if use_excess_returns:
    quantile_portfolios_returns = backtest.compute_excess_returns(
        quantile_portfolios_returns, rf, start_date=start_date
    )
else:
    quantile_portfolios_returns = quantile_portfolios_returns.loc[start_date:]

In [ ]:
rename_map = {
    "Q1_Q1": "EBC_High_Pref_High",
    "Q1_Q2": "EBC_High_Pref_Mid",
    "Q1_Q3": "EBC_High_Pref_Low",
    "Q2_Q1": "EBC_Mid_Pref_High",
    "Q2_Q2": "EBC_Mid_Pref_Mid",
    "Q2_Q3": "EBC_Mid_Pref_Low",
    "Q3_Q1": "EBC_Low_Pref_High",
    "Q3_Q2": "EBC_Low_Pref_Mid",
    "Q3_Q3": "EBC_Low_Pref_Low",
}

quantile_portfolios_returns = quantile_portfolios_returns.rename(columns=rename_map)


In [ ]:
complete_df = full_merged_df.join(
    quantile_portfolios_returns,
    how="inner"
)


In [ ]:
selected_columns = ['CW','EBC_High_Pref_High', 'EBC_High_Pref_Mid',
       'EBC_High_Pref_Low', 'EBC_Mid_Pref_High', 'EBC_Mid_Pref_Mid',
       'EBC_Mid_Pref_Low', 'EBC_Low_Pref_High', 'EBC_Low_Pref_Mid',
       'EBC_Low_Pref_Low']

In [ ]:
# ============================
# PLOTS
# ============================
plots.plot_cum_log_returns(
    complete_df[selected_columns],
    title=f"Cumulative Log Excess Returns ({n_q1}x{n_q2} Cap-Weighted)",
)

In [ ]:
# FM expected-return grid (annualized %)
plots.plot_expected_return_grid(results["pricing"], results["fm_table"], n_q1, n_q2)

In [ ]:
plot_mean_grid(results["mean_grid"], annualize=True)

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parents[0]
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [ ]:
quantile_portfolios_returns.index.name = "date"

In [ ]:
quantile_portfolios_returns.to_csv(DATA_PROCESSED_DIR / "daily_quantile_returns_3x3.csv")